In [4]:

# Experiment 3: End-to-End Automated Data Pipeline (ETL + Train)

import os
import joblib
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


class DataPipeline:
    """A minimal but complete automated ML data pipeline."""

    def __init__(self, artifact_dir="artifacts"):
        self.artifact_dir = artifact_dir
        self.scaler = StandardScaler()
        self.model = RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )

        # Create artifact directory if it does not exist
        os.makedirs(artifact_dir, exist_ok=True)

    def extract(self):
        data = load_iris(as_frame=True)
        df = data.frame

        print(
            f"[EXTRACT] loaded {df.shape[0]} rows x "
            f"{df.shape[1]} cols"
        )

        return df

    def validate(self, df):
        missing = int(df.isnull().sum().sum())

        assert missing == 0, "Data contains missing values!"
        assert "target" in df.columns, "Target column missing!"

        print(
            "[VALIDATE] no missing values, "
            "target present -> OK"
        )

        return df

    def transform(self, df):
        X = df.drop(columns=["target"])
        y = df["target"]

        X_scaled = self.scaler.fit_transform(X)

        print(
            f"[TRANSFORM] standard-scaled "
            f"{X.shape[1]} features"
        )

        return X_scaled, y

    def train(self, X, y):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

        # Train model
        self.model.fit(X_tr, y_tr)

        # Make predictions
        y_pred = self.model.predict(X_te)

        # Calculate accuracy
        acc = accuracy_score(y_te, y_pred)

        print(
            f"[TRAIN] model fitted, "
            f"test accuracy = {acc:.3f}"
        )

        # Classification report
        print("[REPORT] per-class metrics:")
        print(
            classification_report(
                y_te,
                y_pred,
                digits=2
            )
        )

        return acc

    def persist(self):
        model_path = os.path.join(
            self.artifact_dir,
            "model.pkl"
        )

        scaler_path = os.path.join(
            self.artifact_dir,
            "scaler.pkl"
        )

        # Save model and scaler
        joblib.dump(self.model, model_path)
        joblib.dump(self.scaler, scaler_path)

        print(
            f"[PERSIST] saved model -> {model_path}"
        )

        print(
            f"[PERSIST] saved scaler -> {scaler_path}"
        )

    def run(self):
        print("=== Automated Data Pipeline Started ===")

        # Extract
        df = self.extract()

        # Validate
        df = self.validate(df)

        # Transform
        X, y = self.transform(df)

        # Train + Evaluate
        self.train(X, y)

        # Persist
        self.persist()

        print("=== Pipeline Completed Successfully ===")


if __name__ == "__main__":
    pipeline = DataPipeline()
    pipeline.run()

=== Automated Data Pipeline Started ===
[EXTRACT] loaded 150 rows x 5 cols
[VALIDATE] no missing values, target present -> OK
[TRANSFORM] standard-scaled 4 features
[TRAIN] model fitted, test accuracy = 0.933
[REPORT] per-class metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

[PERSIST] saved model -> artifacts/model.pkl
[PERSIST] saved scaler -> artifacts/scaler.pkl
=== Pipeline Completed Successfully ===


In [7]:

# Experiment 6: ML Pipeline with Role-Based Access Control (RBAC)


from functools import wraps
from datetime import datetime


ROLES = {
    "data_engineer": {"extract", "transform"},
    "data_scientist": {"transform", "train", "evaluate"},
    "ml_engineer": {"train", "evaluate", "deploy"},
    "admin": {"extract", "transform", "train", "evaluate", "deploy"},
    "viewer": {"view"},
}



AUDIT_LOG = []



class AccessDenied(Exception):
    """Raised when a role attempts an action it is not permitted to perform."""
    pass


# ---- Audit function ----
def audit(role, permission, allowed):
    entry = {
        "time": datetime.now().strftime("%H:%M:%S"),
        "role": role,
        "action": permission,
        "result": "ALLOW" if allowed else "DENY",
    }

    AUDIT_LOG.append(entry)



def require(permission):
    """Decorator that enforces RBAC for a pipeline stage."""

    def decorator(func):

        @wraps(func)
        def wrapper(user_role, *args, **kwargs):

            # Check whether the role has the required permission
            allowed = permission in ROLES.get(user_role, set())

            # Record access decision
            audit(user_role, permission, allowed)

            # Deny access if permission is missing
            if not allowed:
                raise AccessDenied(
                    f"Role '{user_role}' is NOT authorized for '{permission}'"
                )

            print(f"[AUTH OK ] {user_role:15s} -> {permission}")

            # Execute the actual function
            return func(user_role, *args, **kwargs)

        return wrapper

    return decorator



@require("extract")
def extract(role):
    return "raw_data"


@require("train")
def train(role):
    return "trained_model"


@require("deploy")
def deploy(role):
    return "deployed"



def try_action(fn, role):
    try:
        result = fn(role)
        print(f"[SUCCESS ] {result}")

    except AccessDenied as e:
        print(f"[DENIED ] {e}")



if __name__ == "__main__":

    print("=== ML Pipeline Access Control ===")

    extract("admin")
    train("admin")
    deploy("admin")

    print("\n=== Unauthorized Attempts ===")

    # These should be blocked
    try_action(deploy, "data_scientist")
    try_action(train, "viewer")
    try_action(extract, "ml_engineer")

    
    print("\n=== Audit Log ===")

    for e in AUDIT_LOG:
        print(
            f"{e['time']} "
            f"{e['role']:15s} "
            f"{e['action']:10s} "
            f"{e['result']}"
        )

=== ML Pipeline Access Control ===
[AUTH OK ] admin           -> extract
[AUTH OK ] admin           -> train
[AUTH OK ] admin           -> deploy

=== Unauthorized Attempts ===
[DENIED ] Role 'data_scientist' is NOT authorized for 'deploy'
[DENIED ] Role 'viewer' is NOT authorized for 'train'
[DENIED ] Role 'ml_engineer' is NOT authorized for 'extract'

=== Audit Log ===
10:48:07 admin           extract    ALLOW
10:48:07 admin           train      ALLOW
10:48:07 admin           deploy     ALLOW
10:48:07 data_scientist  deploy     DENY
10:48:07 viewer          train      DENY
10:48:07 ml_engineer     extract    DENY
